# Topic 11 — Coding MLA from Scratch
**DeepSeek from Scratch** · Shiva Kiran Dadishetty

One decoding step of multi-head latent attention: 5 tokens are already in the latent KV cache, a new token arrives,
and we compute its context vector step by step.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## ① Initial setup and parameters

In [2]:
d_model       = 8                      # embedding dimension
kv_latent_dim = 4                      # KV cache (latent) dimension
n_heads       = 2                      # number of heads
d_head        = d_model // n_heads     # 8 / 2 = 4

# input token embedding (1, 8)
x = np.array([[0.1, -0.2, 0.3, -0.4, 0.5, -0.6, 0.7, -0.8]])
# prior context: 5 tokens  "The next day is bright"
print("x:", x, x.shape)

x: [[ 0.1 -0.2  0.3 -0.4  0.5 -0.6  0.7 -0.8]] (1, 8)


## ② Trained weight matrices

In [3]:
W_q = np.array([
    [0.1, 0.2, 0.1, 0.3, 0.1, 0.2, 0.1, 0.3],
    [0.2, 0.1, 0.3, 0.1, 0.2, 0.1, 0.3, 0.1],
    [0.1, 0.3, 0.1, 0.2, 0.1, 0.3, 0.1, 0.2],
    [0.3, 0.1, 0.2, 0.1, 0.3, 0.1, 0.2, 0.1],
    [0.1, 0.2, 0.1, 0.3, 0.1, 0.2, 0.1, 0.3],
    [0.2, 0.1, 0.3, 0.1, 0.2, 0.1, 0.3, 0.1],
    [0.1, 0.3, 0.1, 0.2, 0.1, 0.3, 0.1, 0.2],
    [0.3, 0.1, 0.2, 0.1, 0.3, 0.1, 0.2, 0.1]])     # (8, 8)  query matrix

W_dkv = np.array([
    [ 0.1,  0.2,  0.3,  0.1],
    [-0.2,  0.1,  0.3,  0.2],
    [ 0.3, -0.1,  0.2,  0.1],
    [ 0.2,  0.3, -0.1,  0.2],
    [ 0.1,  0.2,  0.1,  0.3],
    [-0.1,  0.2,  0.3,  0.1],
    [ 0.3,  0.1,  0.2, -0.1],
    [ 0.2, -0.1,  0.3,  0.2]])                      # (8, 4)  down-projection

W_uk = np.array([
    [0.2, 0.1, 0.3, 0.1, 0.2, 0.1, 0.3, 0.1],
    [0.3, 0.2, 0.1, 0.2, 0.3, 0.2, 0.1, 0.2],
    [0.1, 0.3, 0.2, 0.3, 0.1, 0.3, 0.2, 0.3],
    [0.2, 0.1, 0.3, 0.1, 0.2, 0.1, 0.3, 0.1]])     # (4, 8)  up-projection for keys

W_uv = np.array([
    [0.2, 0.1, 0.3, 0.1, 0.2, 0.1, 0.3, 0.1],
    [0.1, 0.2, 0.1, 0.3, 0.1, 0.2, 0.1, 0.3],
    [0.3, 0.1, 0.2, 0.1, 0.3, 0.1, 0.2, 0.1],
    [0.1, 0.3, 0.1, 0.2, 0.1, 0.3, 0.1, 0.2]])     # (4, 8)  up-projection for values

W_o = np.array([
    [0.1, 0.2, 0.3, 0.1, 0.2, 0.1, 0.3, 0.1],
    [0.2, 0.1, 0.2, 0.3, 0.1, 0.2, 0.1, 0.3],
    [0.3, 0.2, 0.1, 0.2, 0.3, 0.1, 0.2, 0.1],
    [0.1, 0.3, 0.2, 0.1, 0.2, 0.3, 0.1, 0.2],
    [0.2, 0.1, 0.3, 0.2, 0.1, 0.2, 0.3, 0.1],
    [0.1, 0.2, 0.1, 0.3, 0.2, 0.1, 0.2, 0.3],
    [0.3, 0.1, 0.2, 0.1, 0.3, 0.2, 0.1, 0.2],
    [0.2, 0.3, 0.1, 0.2, 0.1, 0.3, 0.2, 0.1]])     # (8, 8)  output projection
# W_o[7, 0] is hidden in the source screenshot; 0.2 is a placeholder.

## ③ Latent KV cache computed previously
$X \cdot W_{dkv} \rightarrow (5, 4)$, with $X$ of shape (5, 8) and $W_{dkv}$ of shape (8, 4).

In [4]:
previous_cache = np.array([
    [ 1.2,  0.3, -0.5, -0.8],    # The
    [ 0.9, -0.2,  0.6,  0.4],    # next
    [-0.7,  0.5,  0.8, -0.3],    # day
    [ 0.6,  0.7, -0.4,  0.1],    # is
    [-0.5, -0.6,  0.3,  0.9]])   # bright
print("previous_cache:", previous_cache.shape)

previous_cache: (5, 4)


## ④ Absorbed query matrix
For each head, slice the head's columns of $W_q$ and $W_{uk}$ first, then multiply:
head 1: $W_q[:, 0{:}4]$ (8, 4) · $W_{uk}[:, 0{:}4]^\top$ (4, 4) → (8, 4); head 2 uses columns 4–7.
Result: (n_heads, d_model, latent_dim) = (2, 8, 4).

In [5]:
absorbed_qk = np.stack([
    W_q[:, h*d_head:(h+1)*d_head] @ W_uk[:, h*d_head:(h+1)*d_head].T
    for h in range(n_heads)])                              # (2, 8, 4)
for h in range(n_heads):
    print(f"absorbed_qk[{h}] (8x4):  head {h+1}\n", absorbed_qk[h])

absorbed_qk[0] (8x4):  head 1
 [[0.1  0.14 0.18 0.1 ]
 [0.15 0.13 0.14 0.15]
 [0.1  0.14 0.18 0.1 ]
 [0.14 0.15 0.13 0.14]
 [0.1  0.14 0.18 0.1 ]
 [0.15 0.13 0.14 0.15]
 [0.1  0.14 0.18 0.1 ]
 [0.14 0.15 0.13 0.14]]
absorbed_qk[1] (8x4):  head 2
 [[0.1  0.14 0.18 0.1 ]
 [0.15 0.13 0.14 0.15]
 [0.1  0.14 0.18 0.1 ]
 [0.14 0.15 0.13 0.14]
 [0.1  0.14 0.18 0.1 ]
 [0.15 0.13 0.14 0.15]
 [0.1  0.14 0.18 0.1 ]
 [0.14 0.15 0.13 0.14]]


## ⑤ Compute compressed KV representation for the new token
$x \cdot W_{dkv}$: (1, 8) · (8, 4) → (1, 4), then LayerNorm (as in the PyTorch code below).

In [6]:
def layer_norm(v, eps=1e-5):
    return (v - v.mean(axis=-1, keepdims=True)) / np.sqrt(v.var(axis=-1, keepdims=True) + eps)

c_raw    = x @ W_dkv            # (1, 4)
new_c_kv = layer_norm(c_raw)    # (1, 4)
print("new_c_kv (before layernorm):", c_raw, c_raw.shape)
print("new_c_kv (after layernorm) :", new_c_kv, new_c_kv.shape)

new_c_kv (before layernorm): [[ 0.22 -0.02 -0.16 -0.22]] (1, 4)
new_c_kv (after layernorm) : [[ 1.5647  0.1476 -0.679  -1.0333]] (1, 4)


## ⑥ Append to KV cache
Old cache (5, 4) → new cache (6, 4).

In [7]:
c_kv = np.vstack([previous_cache, new_c_kv])
print("c_kv:", c_kv.shape)
print(c_kv)

c_kv: (6, 4)
[[ 1.2     0.3    -0.5    -0.8   ]
 [ 0.9    -0.2     0.6     0.4   ]
 [-0.7     0.5     0.8    -0.3   ]
 [ 0.6     0.7    -0.4     0.1   ]
 [-0.5    -0.6     0.3     0.9   ]
 [ 1.5647  0.1476 -0.679  -1.0333]]


## ⑦ Compute query representation for the new token + attention
Both heads use the whole new token x (1, 8). For each head: absorbed query = x · absorbed_qk[h], (1, 8)·(8, 4) → (1, 4);
attention scores = absorbed query · $c_{kv}^\top$ / $\sqrt{d_{head}}$ → (1, 6); attention weights = softmax(scores) → (1, 6).

In [8]:
def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

print("c_kv transposed (4x6):\n", c_kv.T)
attn_weights = []
for h in range(n_heads):
    tmp     = x @ absorbed_qk[h]                          # (1, 4)  absorbed query vector
    scores  = tmp @ c_kv.T / np.sqrt(d_head)              # (1, 6)
    weights = softmax(scores)                             # (1, 6)
    attn_weights.append(weights)
    print(f"head {h+1}  absorbed query   :", tmp, tmp.shape)
    print(f"head {h+1}  attention scores :", scores, scores.shape)
    print(f"head {h+1}  attention weights:", weights, weights.shape)

c_kv transposed (4x6):
 [[ 1.2     0.9    -0.7     0.6    -0.5     1.5647]
 [ 0.3    -0.2     0.5     0.7    -0.6     0.1476]
 [-0.5     0.6     0.8    -0.4     0.3    -0.679 ]
 [-0.8     0.4    -0.3     0.1     0.9    -1.0333]]
head 1  absorbed query   : [[-0.128 -0.06   0.02  -0.128]] (1, 4)
head 1  attention scores : [[-0.0396 -0.0712  0.057  -0.0698 -0.0046 -0.0452]] (1, 6)
head 1  attention weights: [[0.1647 0.1596 0.1814 0.1598 0.1706 0.1638]] (1, 6)
head 2  absorbed query   : [[-0.128 -0.06   0.02  -0.128]] (1, 4)
head 2  attention scores : [[-0.0396 -0.0712  0.057  -0.0698 -0.0046 -0.0452]] (1, 6)
head 2  attention weights: [[0.1647 0.1596 0.1814 0.1598 0.1706 0.1638]] (1, 6)


## ⑧ Find values matrix
Updated cache (6, 4) · $W_{uv}$ (4, 8) → V (6, 8). Columns 0–3 → head 1, columns 4–7 → head 2, each (6, 4).

In [9]:
V = c_kv @ W_uv                                          # (6, 8)
V_heads = [V[:, h*d_head:(h+1)*d_head] for h in range(n_heads)]   # 2 x (6, 4)
print("V:", V.shape)
print(V)

V: (6, 8)
[[ 0.04   -0.11    0.21   -0.      0.04   -0.11    0.21   -0.    ]
 [ 0.38    0.23    0.41    0.17    0.38    0.23    0.41    0.17  ]
 [ 0.12    0.02   -0.03    0.1     0.12    0.02   -0.03    0.1   ]
 [ 0.08    0.19    0.18    0.25    0.08    0.19    0.18    0.25  ]
 [ 0.02    0.13   -0.06   -0.02    0.02    0.13   -0.06   -0.02  ]
 [ 0.0207 -0.1919  0.245  -0.0738  0.0207 -0.1919  0.245  -0.0738]]


## ⑨ Find context vector
Each head: attention weights (1, 6) · V (6, 4) = (1, 4). Concatenate → (1, 8).

In [10]:
context_heads    = [attn_weights[h] @ V_heads[h] for h in range(n_heads)]
combined_context = np.hstack(context_heads)               # (1, 8)
for h in range(n_heads):
    print(f"context head {h+1}:", context_heads[h], context_heads[h].shape)
print("combined_context:", combined_context, combined_context.shape)

context head 1: [[0.1086 0.0433 0.1533 0.0697]] (1, 4)
context head 2: [[0.1086 0.0433 0.1533 0.0697]] (1, 4)
combined_context: [[0.1086 0.0433 0.1533 0.0697 0.1086 0.0433 0.1533 0.0697]] (1, 8)


## ⑩ Output projection
combined_context (1, 8) · $W_o$ (8, 8) → (1, 8), used to compute the logits for next-token prediction.

In [11]:
output = combined_context @ W_o                           # (1, 8)
print("output:", output.shape)

output: (1, 8)


## Check — absorbed MLA = unabsorbed MLA
Build the keys explicitly (K = c_kv · W_uk) and compare the attention scores and context vectors.

In [12]:
K = c_kv @ W_uk                     # (6, 8)
Q = x @ W_q                         # (1, 8)
for h in range(n_heads):
    cols = slice(h*d_head, (h+1)*d_head)
    s_explicit = Q[:, cols] @ K[:, cols].T / np.sqrt(d_head)
    s_absorbed = (x @ absorbed_qk[h]) @ c_kv.T / np.sqrt(d_head)
    print(f"head {h+1}: max |score difference| =", np.abs(s_explicit - s_absorbed).max())
print("cache numbers  MLA:", c_kv.size, "  MHA:", 2 * c_kv.shape[0] * d_model)

head 1: max |score difference| = 4.163336342344337e-17
head 2: max |score difference| = 4.163336342344337e-17
cache numbers  MLA: 24   MHA: 96


# Part 2 — PyTorch implementation
Run these cells in Colab (PyTorch and seaborn required).

## Step 1: Writing the code for the Multi-Head Latent Attention

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RopelessMLA(nn.Module):
    def __init__(self, d_model, n_heads, kv_latent_dim):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.dh = d_model // n_heads                                # dimension per head

        # Projection layers
        self.W_q   = nn.Linear(d_model, d_model, bias=False)        # Query projection
        self.W_dkv = nn.Linear(d_model, kv_latent_dim, bias=False)  # Compress into latent KV space
        self.W_uk  = nn.Linear(kv_latent_dim, d_model, bias=False)  # Decompress K
        self.W_uv  = nn.Linear(kv_latent_dim, d_model, bias=False)  # Decompress V
        self.W_o   = nn.Linear(d_model, d_model, bias=False)        # Final output projection

        self.ln = nn.LayerNorm(kv_latent_dim)
        self.register_buffer('absorbed_k', None)                    # Holds W_q @ W_uk

    def forward(self, x, kv_cache=None, past_length=0):
        B, S, D = x.size()

        # Compute absorbed_k once: one (D, latent_dim) matrix per head
        # (nn.Linear.weight is (out_features, in_features); rows h*dh:(h+1)*dh belong to head h).
        # During training W_q and W_uk change, so recompute it every step (cache only in eval mode).
        if self.absorbed_k is None or self.training:
            w_q  = self.W_q.weight.view(self.n_heads, self.dh, self.d_model)    # (n_heads, dh, D)
            w_uk = self.W_uk.weight.view(self.n_heads, self.dh, -1)             # (n_heads, dh, latent_dim)
            self.absorbed_k = torch.bmm(w_q.transpose(1, 2), w_uk)              # (n_heads, D, latent_dim)

        # Compress x into latent KV space
        new_c_kv = self.ln(self.W_dkv(x))                           # (B, S, latent_dim)
        if kv_cache is None:
            c_kv = new_c_kv
        else:
            c_kv = torch.cat([kv_cache, new_c_kv], dim=1)           # (B, S_total, latent_dim)

        S_full = c_kv.size(1)

        # Decompress V to full d_model and split into heads
        v_full = self.W_uv(c_kv)                                    # (B, S_full, D)
        v = v_full.view(B, S_full, self.n_heads, self.dh).transpose(1, 2)
                                                                    # (B, n_heads, S_full, dh)

        # Compute attention scores: every head uses the full x (W_q is absorbed)
        attn_scores = torch.zeros(B, self.n_heads, S, S_full, device=x.device)
        for h in range(self.n_heads):
            tmp = torch.matmul(x, self.absorbed_k[h])               # (B, S, latent_dim)
            attn_scores[:, h] = torch.bmm(tmp, c_kv.transpose(1, 2))    # (B, S, S_full)

        # Scale and apply causal mask
        attn_scores = attn_scores / (self.dh ** 0.5)
        mask = torch.tril(torch.ones((S, S_full), device=x.device), diagonal=past_length)
        attn_scores = attn_scores.masked_fill(mask.view(1, 1, S, S_full) == 0,
                                              float('-inf'))

        # Softmax to get attention weights
        attn_weights = F.softmax(attn_scores, dim=-1)               # (B, n_heads, S, S_full)

        # Apply attention weights to each head's V separately
        out_heads = []
        for h in range(self.n_heads):
            context_h = torch.matmul(attn_weights[:, h], v[:, h])   # (B, S, dh)
            out_heads.append(context_h)

        # Concatenate all head outputs along the feature dimension
        out = torch.cat(out_heads, dim=-1)                          # (B, S, D)

        return self.W_o(out), c_kv                       # Final output projection + updated cache

## Step 2: Memory testing

In [ ]:
# === Demo for Testing ===
def demo():
    model = RopelessMLA(d_model=512, n_heads=8, kv_latent_dim=256)
    x = torch.randn(1, 5, 512)

    out, cache = model(x)
    print(f"Output: {out.shape}, Cache: {cache.shape}")

    # Memory comparison
    B, T = 1, 5
    std_size = B * 2 * T * 512 * 4 / 1024     # KB (standard KV: B * 2 (K,V) * T * D * float32)
    latent_size = B * T * 256 * 4 / 1024      # KB (latent cache: B * T * latent_dim * float32)
    print(f"Memory: Standard={std_size:.1f}KB, Latent={latent_size:.1f}KB, "
          f"Reduction={std_size/latent_size:.1f}x")

if __name__ == "__main__":
    demo()

## Step 3: Cache testing - single new inference

In [ ]:
def demo_cache_usage():
    torch.manual_seed(0)
    model = RopelessMLA(d_model=8, n_heads=2, kv_latent_dim=4)

    # ---- Step 1: Initial input (sequence of 5 tokens) ----
    x1 = torch.randn(1, 5, 8)                                  # (B=1, S=5, D=8)
    out1, cache1 = model(x1)
    print("Step 1: Initial input")
    print(f"Output shape: {out1.shape}")
    print(f"Cache shape:  {cache1.shape}")                     # Expect: (1, 5, 4)

    # ---- Step 2: Append 1 token ----
    x2 = torch.randn(1, 1, 8)                                  # (B=1, S=1, D=8)
    out2, cache2 = model(x2, kv_cache=cache1, past_length=5)
    print("\nStep 2: New token")
    print(f"Output shape: {out2.shape}")
    print(f"Cache shape:  {cache2.shape}")                     # Expect: (1, 6, 4)

demo_cache_usage()

## Step 4: Cache growth

In [ ]:
def demo_kv_cache_growth(num_initial_tokens=5, num_new_tokens=3):
    torch.manual_seed(0)
    model = RopelessMLA(d_model=8, n_heads=2, kv_latent_dim=4)

    # Step 1: Start with initial token batch
    x = torch.randn(1, num_initial_tokens, 8)
    out, cache = model(x)
    print(f"Step 0: Initial input of {num_initial_tokens} tokens -> "
          f"cache shape: {cache.shape}")

    # Step 2: Incrementally append new tokens one at a time
    for step in range(1, num_new_tokens + 1):
        new_token = torch.randn(1, 1, 8)                            # (B=1, S=1, D=8)
        out, cache = model(new_token, kv_cache=cache, past_length=cache.shape[1])
        print(f"Step {step}: Added 1 token -> cache shape: {cache.shape}")

demo_kv_cache_growth(num_initial_tokens=50, num_new_tokens=4)

## Step 5: Visualizing W_q, W_uk and W_uv matrices for different heads

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

class RopelessMLA(nn.Module):
    def __init__(self, d_model=8, n_heads=2, kv_latent_dim=4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.dh = d_model // n_heads

        self.W_q  = nn.Linear(d_model, d_model, bias=False)
        self.W_uk = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.W_uv = nn.Linear(kv_latent_dim, d_model, bias=False)

model = RopelessMLA()

# Extract weights
W_q  = model.W_q.weight.detach().numpy()
W_uk = model.W_uk.weight.detach().numpy()
W_uv = model.W_uv.weight.detach().numpy()

# Split per head
n_heads = model.n_heads
dh = model.dh
W_q_heads  = W_q.reshape(n_heads, dh, -1)      # (n_heads, dh, D)
W_uk_heads = W_uk.reshape(n_heads, dh, -1)     # (n_heads, dh, latent_dim)
W_uv_heads = W_uv.reshape(n_heads, dh, -1)     # (n_heads, dh, latent_dim)

# Plot heatmaps
fig, axes = plt.subplots(n_heads, 3, figsize=(12, 5))
for h in range(n_heads):
    sns.heatmap(W_q_heads[h], ax=axes[h, 0], cbar=False, annot=True, fmt=".2f")
    axes[h, 0].set_title(f"Head {h} - W_q")

    sns.heatmap(W_uk_heads[h], ax=axes[h, 1], cbar=False, annot=True, fmt=".2f")
    axes[h, 1].set_title(f"Head {h} - W_uk")

    sns.heatmap(W_uv_heads[h], ax=axes[h, 2], cbar=False, annot=True, fmt=".2f")
    axes[h, 2].set_title(f"Head {h} - W_uv")

plt.tight_layout()
plt.show()